# DLR Geocoding Services

## Goal

Retrieve candidate locations from DLR GeoNames and Photon services.

## What you will do

- Build service URLs.
- Call services through Python.
- Compare candidate tables.
- Save candidates to `outputs/results/geocoder_candidates.csv`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import GEONAMES_BASE_URL, PHOTON_BASE_URL, RESULTS_DIR
from src.data_utils import save_dataframe
from src.geocoder_clients import geocode_geonames, geocode_photon, compare_geocoders

## Step 1: Browser URL examples

In [ ]:
print('GeoNames:', f'{GEONAMES_BASE_URL}?location=Berlin')
print('Photon:', f'{PHOTON_BASE_URL}?q=Berlin&limit=5')

## Step 2: Query examples

In [ ]:
queries = ['Berlin', 'Paris', 'Cambridge', 'Springfield', 'Izmir', 'Bayern', 'Rhine', 'Danube']
queries

## Step 3: Call GeoNames defensively

In [ ]:
geonames_berlin = geocode_geonames('Berlin', limit=5)
geonames_berlin

## Step 4: Call Photon defensively

In [ ]:
photon_berlin = geocode_photon('Berlin', limit=5)
photon_berlin

## Step 5: Compare candidates

Different geocoders may return different JSON structures and ranking orders.

In [ ]:
frames = []
for query in queries:
    frames.append(compare_geocoders(query, limit=5))
candidates = pd.concat([f for f in frames if not f.empty], ignore_index=True) if any(not f.empty for f in frames) else pd.DataFrame()

## Step 6: Use built-in candidates if services are unavailable

In [ ]:
if candidates.empty:
    candidates = pd.DataFrame([
        {'query': 'Berlin', 'source': 'mock', 'name': 'Berlin', 'country': 'Germany', 'state': 'Berlin', 'county': '', 'lat': 52.52, 'lon': 13.405, 'feature_class': 'P', 'feature_code': 'PPLC', 'population': 3769000, 'raw': '{}'},
        {'query': 'Paris', 'source': 'mock', 'name': 'Paris', 'country': 'France', 'state': 'Ile-de-France', 'county': '', 'lat': 48.8566, 'lon': 2.3522, 'feature_class': 'P', 'feature_code': 'PPLC', 'population': 2140526, 'raw': '{}'},
    ])
candidates.head(10)

## Step 7: Save candidate results

These candidates are not final toponym resolution yet.

In [ ]:
out = save_dataframe(candidates, RESULTS_DIR / 'geocoder_candidates.csv')
print('Saved:', out)

## Exercise

Try an ambiguous place name such as `Springfield`, `Cambridge`, or a local place from your own data.

In [ ]:
exercise_query = 'Cambridge'
compare_geocoders(exercise_query, limit=5)

## Common issues

- Internal endpoints may require DLR network or VPN.
- Raw JSON is preserved for debugging.
- Missing fields are expected; parsers should not assume every service returns the same schema.